In [ ]:
# df['WaterVar']=df['Total Body Water (TBW)']-df['Extracellular Water (ECW)']-df['Intracellular Water (ICW)']
# df['area/mass']= df['Visceral Muscle Area (VMA) (Kg)']/df['Muscle Mass (MM)']
# df['sad'] = (df['Extracellular Water (ECW)']/df['Total Body Water (TBW)']*100) -df['Extracellular Fluid/Total Body Water (ECF/TBW)']

In [ ]:
def cal_Obesity(BMI):
    return abs((BMI / 22 - 1) * 100)

In [ ]:
# 1. DATA PREP
df_orig = df.copy()
df_orig['Calculated_Obesity'] = df_orig['Body Mass Index (BMI)'].apply(cal_Obesity)
df_clean = df_orig[~np.isclose(df_orig['Obesity (%)'], 1954.00)].copy()

# --- METRIC 1: REGRESSION SCORE (Consistency) ---
# We square the Pearson R to get R^2 for the regression line
r_orig, _ = pearsonr(df_orig['Calculated_Obesity'], df_orig['Obesity (%)'])
r2_regr_orig = r_orig**2

r_clean, _ = pearsonr(df_clean['Calculated_Obesity'], df_clean['Obesity (%)'])
r2_regr_clean = r_clean**2

# --- METRIC 2: THEORETICAL SCORE (Accuracy) ---
# How well does it fit the PHYSICS FORMULA (y=x)?
def calc_theoretical_r2(df_input):
    # Order matters: (True, Pred) -> (Sensor, Formula)
    # This measures how well the formula predicts the sensor
    return r2_score(df_input['Obesity (%)'], df_input['Calculated_Obesity'])

r2_theory_orig = calc_theoretical_r2(df_orig)
r2_theory_clean = calc_theoretical_r2(df_clean)

# 2. VISUALIZATION
fig, axes = plt.subplots(2, 2, figsize=(16, 12), layout='constrained')

LABEL_Y_SENSOR = "Measured Obesity Degree (Sensor)"
LABEL_X_FORMULA = "Calculated Obesity Degree (Formula)"

# ==========================================
# ROW 1: SCENARIO A (Original)
# ==========================================
# 1A. Boxplot
sns.boxplot(data=df_orig[['Obesity (%)']], orient='h', ax=axes[0, 0], color='#C0392B', width=0.4)
format_plot(axes[0, 0], 'A1. Sensor Distribution (With Error)', LABEL_Y_SENSOR, '',True)

# 1B. Regression
sns.regplot(
    data=df_orig, x='Calculated_Obesity', y='Obesity (%)', ax=axes[0, 1],
    color='#C0392B', scatter_kws={'alpha': 0.5, 's': 30},
    line_kws={'color': 'black', 'label': 'Regression Trend'}
)
# Outlier Annotation
outlier_row = df_orig[np.isclose(df_orig['Obesity (%)'], 1954.00)]
if not outlier_row.empty:
    x_out, y_out = outlier_row['Calculated_Obesity'].values[0], outlier_row['Obesity (%)'].values[0]
    axes[0, 1].annotate("Sensor Error", xy=(x_out, y_out), xytext=(x_out-40, y_out-300),
                        arrowprops=dict(facecolor='black', shrink=0.05), fontsize=10,
                        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", alpha=0.9))

# [CHANGE] Unified R² Metrics
title_a = (f"A2. Original Correlation: FAILED\n"
           f"Regression Fit ($R^2$): {r2_regr_orig:.4f}  |  Theoretical Match ($R^2$): {r2_theory_orig:.4f}")
format_plot(axes[0, 1], title_a, LABEL_X_FORMULA, LABEL_Y_SENSOR)


# ==========================================
# ROW 2: SCENARIO B (Cleaned)
# ==========================================
# 2A. Boxplot
sns.boxplot(data=df_clean[['Obesity (%)']], orient='h', ax=axes[1, 0], color='#5DADE2', width=0.4)
format_plot(axes[1, 0], 'B1. Sensor Distribution (Cleaned)', LABEL_Y_SENSOR, '',True)

# 2B. Regression
sns.regplot(
    data=df_clean, x='Calculated_Obesity', y='Obesity (%)', ax=axes[1, 1],
    color='teal', scatter_kws={'alpha': 0.5, 's': 30},
    line_kws={'color': '#E67E22', 'linewidth': 3, 'label': 'Regression Trend'}
)
limit = max(df_clean['Obesity (%)'].max(), df_clean['Calculated_Obesity'].max())
axes[1, 1].plot([0, limit], [0, limit], color='gray', linestyle='--', linewidth=1.5, label='Theoretical Match (y=x)')

# [CHANGE] Unified R² Metrics
title_b = (f"B2. Cleaned Correlation: CONFIRMED\n"
           f"Regression Fit ($R^2$): {r2_regr_clean:.4f}  |  Theoretical Match ($R^2$): {r2_theory_clean:.4f}")
format_plot(axes[1, 1], title_b, LABEL_X_FORMULA, LABEL_Y_SENSOR)

axes[1, 1].legend(loc='lower right')

plt.show()